In [ ]:
# 1. Install dependencies
!pip install ultralytics -q

import os
import yaml
from ultralytics import YOLO
import torch


In [ ]:
dataset_dir = "/kaggle/input/pcb-defect-dataset/pcb-defect-dataset"
if not os.path.exists(dataset_dir):
    raise FileNotFoundError(
        "Dataset directory not found. Please add PCB-DEFECT-DATASET to your Kaggle Notebook Datasets."
    )

# Load original data.yaml
data_yaml_path = os.path.join(dataset_dir, "data.yaml")
with open(data_yaml_path, 'r') as f:
    orig_config = yaml.safe_load(f)

# Extract and construct absolute paths
names = orig_config.get('names')
if names is None:
    raise KeyError("'names' key not found in data.yaml. Check your data configuration.")
nc = orig_config.get('nc', len(names))

# Original paths (relative to data.yaml): e.g., '../images/train'
train_rel = orig_config.get('train')
val_rel = orig_config.get('val', orig_config.get('test'))
if not train_rel or not val_rel:
    raise KeyError("'train' or 'val' split not defined in data.yaml.")

train_path = os.path.join(dataset_dir, train_rel.replace('../', ''))
val_path   = os.path.join(dataset_dir, val_rel.replace('../', ''))
assert os.path.isdir(train_path), f"Train directory not found: {train_path}"
assert os.path.isdir(val_path),   f"Val directory not found:   {val_path}"

# Create an absolute-path data yaml for YOLO
abs_yaml = {
    'train': train_path,
    'val':   val_path,
    'nc':    nc,
    'names': names
}
abs_yaml_path = '/kaggle/working/data_abs.yaml'
with open(abs_yaml_path, 'w') as f:
    yaml.dump(abs_yaml, f)
print(f"Created absolute-path data.yaml at {abs_yaml_path}")

In [ ]:


# 3. Environment Check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 4. Model Training
model = YOLO('yolov8s.pt').to(device)

# Hyperparameters
epochs = 30
imgsz = 640
batch = 64     # adjust if OOM
patience = 10   # early stopping
run_name = 'pcb_defect_yolov8s_kaggle'

print("Starting training...")
results = model.train(
    data=abs_yaml_path,
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    patience=patience,
    device=device,
    name=run_name
)
print(f"Training finished. Results saved to: {results.save_dir}")

In [ ]:
# 5. Validation
best_weights = os.path.join(results.save_dir, 'weights', 'best.pt')
if os.path.exists(best_weights):
    best_model = YOLO(best_weights).to(device)
    print("Running validation on best.pt...")
    metrics = best_model.val(data=abs_yaml_path, imgsz=imgsz, batch=batch)
    try:
        print(f"mAP50-95: {metrics.box.map:.4f}")
        print(f"mAP50:   {metrics.box.map50:.4f}")
    except Exception:
        print("Could not extract standard box metrics from validation output.")
else:
    print("Best weights not found. Skipping validation.")


In [ ]:

# 1. Install dependencies
!pip install ultralytics -q

import os
import yaml
from ultralytics import YOLO
import torch
import matplotlib.pyplot as plt
from PIL import Image

# 2. Kaggle Dataset Setup

dataset_dir = "/kaggle/input/pcb-defect-dataset/pcb-defect-dataset"
if not os.path.exists(dataset_dir):
    raise FileNotFoundError(
        "Dataset directory not found. Add PCB-DEFECT-DATASET to your Kaggle Notebook Datasets."
    )

# Load original data.yaml
data_yaml_path = os.path.join(dataset_dir, "data.yaml")
with open(data_yaml_path, 'r') as f:
    orig = yaml.safe_load(f)

# Extract classes
names = orig.get('names')
if names is None:
    raise KeyError("'names' key missing in data.yaml.")
nc = orig.get('nc', len(names))

# Build absolute paths helper
def abs_dir(rel): return os.path.join(dataset_dir, rel.replace('../', ''))
train_path = abs_dir(orig['train'])
val_path   = abs_dir(orig.get('val', orig.get('test')))
test_path  = abs_dir(orig.get('test', orig.get('val')))
for p in [train_path, val_path, test_path]:
    if not (p and os.path.isdir(p)):
        print(f"Warning: directory not found: {p}")

# Create absolute-path data yaml for training/validation
data_cfg = {'train': train_path, 'val': val_path, 'nc': nc, 'names': names}
abs_yaml_path = '/kaggle/working/data_abs.yaml'
with open(abs_yaml_path, 'w') as f: yaml.dump(data_cfg, f)
print(f"Data config written at {abs_yaml_path}")

# 3. Environment & Model Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Choose model variant: nano(smallest), small, medium, large\ n# e.g., 'yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt'
model_variant = 'yolov8m.pt'  # upgrade to medium for better accuracy
model = YOLO(model_variant).to(device)

# 4. Training Hyperparameters & Augmentations
epochs = 30               # increase epochs
imgsz = 640
batch = 32
patience = 15             # early stopping patience
augment = True            # enable built-in augmentations
optimizer = 'AdamW'
lr0 = 0.001
run_name = 'pcb_yolov8m_aug'

# 5. Training
print(f"Training {model_variant} with augment={augment}...")
results = model.train(
    data=abs_yaml_path,
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    patience=patience,
    device=device,
    name=run_name,
    augment=augment,
    optimizer=optimizer,
    lr0=lr0
)
print(f"Finished training. Results at: {results.save_dir}")

# 6. Validation
best = os.path.join(results.save_dir, 'weights', 'best.pt')
if os.path.exists(best):
    m = YOLO(best).to(device)
    print("Validation metrics:")
    stats = m.val(data=abs_yaml_path, imgsz=imgsz, batch=batch)
    try:
        print(f"mAP50-95: {stats.box.map:.4f}")
        print(f"mAP50:   {stats.box.map50:.4f}")
    except:
        print(stats)
    if stats.box.map50 < 0.70:
        print("mAP50 < 0.70. Consider further tuning or adding more data.")
else:
    print("Best weights not found. Skipping validation.")

# 7. Inference on Test or Validation Set
infer_dir = test_path if (test_path and os.listdir(test_path)) else val_path
print(f"Running inference on: {infer_dir}")

out_dir = os.path.join(results.save_dir, 'inference_outputs')
os.makedirs(out_dir, exist_ok=True)

# Visualize a few samples
top_k = 5
imgs = os.listdir(infer_dir)[:top_k]
for i, fn in enumerate(imgs):
    ip = os.path.join(infer_dir, fn)
    res = m.predict(source=ip, imgsz=imgsz, conf=0.5)[0]
    img = Image.open(ip).convert('RGB')
    plt.figure(figsize=(8,8))
    plt.imshow(img)
    ax = plt.gca()
    for box in res.boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        cls, conf = int(box.cls[0]), box.conf[0]
        rect = plt.Rectangle((x1,y1), x2-x1, y2-y1, fill=False, linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{names[cls]} {conf:.2f}", fontsize=12,
                color='yellow', backgroundcolor='black')
    plt.axis('off')
    plt.title(f"Inference {i+1}: {fn}")
    plt.savefig(os.path.join(out_dir, f"inf_{i+1}_{fn}"))
    display(plt.show())

print(f"Saved inference outputs to {out_dir}")
